# TPCx-AI UC9: Image Classification with Distributed PyTorch

Multi-class image classification using ResNet-50 on the TPCx-AI benchmark (unofficial).

**Pipeline:** Unity Catalog Volume (FUSE) → PyTorch DataLoader → DDP Training

## Prerequisites

- A cluster running **Databricks Runtime for Machine Learning** (GPU) with PyTorch support.
- Workers: `g5.12xlarge` (4x A10G GPUs, 48 vCPU, 192 GB RAM) or equivalent GPU instance.
- **Cluster setup:**
  - Set `spark.task.resource.gpu.amount = 4` in Spark config (Advanced options)
- **Notebook variables (configured in code cells below):**
  - `NUM_PROCESSES` should match GPUs per node (4 for g5.12xlarge)
  - For multi-node: set `LOCAL_MODE = False`; for single-node: set `LOCAL_MODE = True`
- **Dataset:** TPCx-AI UC9 image classification dataset loaded into Unity Catalog managed volume.

  **Setup the dataset:**
  1. Generate images locally using TPCx-AI PDGF: [TPCx-AI Specification](https://www.tpc.org/tpc_documents_current_versions/current_specifications5.asp)
  2. Upload to S3 bucket
  3. Run `setup_dbx.ipynb` to load into managed volume
  4. The config cell below shows the expected path format from setup

## Running

1. Attach this notebook to your GPU cluster.
2. In the config cell, adjust `IMAGE_ROOT`, `LOCAL_MODE`, and training hyperparameters.
3. Run all cells. Training time and loss are printed inline.

## Pipeline Components

- Unity Catalog Volume — stores images in cloud object storage with FUSE mount for direct file access
- Spark `binaryFile` reader — discovers all image paths without loading pixel data
- `PathDataset` — lazy-loads images from FUSE mount in `__getitem__`
- `DistributedSampler` — shards data across all GPU ranks (multi-node aware)
- `TorchDistributor` — Databricks wrapper for launching PyTorch DDP training across cluster
- PyTorch DDP — synchronizes gradients across all GPUs via NCCL

In [ ]:
import os
from datetime import datetime as dt
from pyspark.ml.torch.distributor import TorchDistributor

In [ ]:
print(f"Driver: {spark.conf.get('spark.driver.host')}")
print(f"Workers: {spark.conf.get('spark.databricks.clusterUsageTags.clusterWorkers')}")
print(f"spark.task.resource.gpu.amount: {spark.sparkContext.getConf().get('spark.task.resource.gpu.amount', 'not set')}")

In [ ]:
# ---- Data ----
IMAGE_ROOT = "/Volumes/tpcxai_uc9/public/uc9_images/CUSTOMER_IMAGES/"  # Path from setup_dbx.ipynb
OUTPUT_DIR = "/local_disk0/model_output"                                 # Final model save path
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- Cluster ----
NUM_PROCESSES = 4          # GPUs per node (g5.12xlarge = 4 x A10G)
LOCAL_MODE = False         # True for single-node training, False for multi-node
spark.conf.set("spark.task.resource.gpu.amount", str(NUM_PROCESSES))

# ---- Training ----
EPOCHS        = 10
BATCH_SIZE    = 256
LEARNING_RATE = 1e-3
NUM_WORKERS   = 8          # DataLoader workers per GPU rank
PREFETCH      = 4          # Prefetch batches per DataLoader worker
SEED          = 42

## Discover Images

Lists all image paths from the Unity Catalog Volume and builds the class-to-index mapping. Runs on the driver; paths and labels are passed to each worker via `TorchDistributor`.

In [ ]:
raw = (
    spark.read.format("binaryFile")
         .option("pathGlobFilter", "*.png")
         .option("recursiveFileLookup", "true")
         .load(IMAGE_ROOT)
         .select("path")
         .collect()
)

all_paths = sorted(r.path.replace("dbfs:", "") for r in raw)
assert len(all_paths) > 0, f"No PNG files found under {IMAGE_ROOT}"

class_names  = sorted({p.rsplit("/", 2)[-2] for p in all_paths})
class_to_idx = {name: i for i, name in enumerate(class_names)}

print(f"Discovered {len(all_paths):,} images across {len(class_to_idx)} classes")

## Training Function

Runs on each GPU worker. Reads images lazily from FUSE, applies transforms, and trains ResNet-50 with DDP.

- **PathDataset**: Lazy image loading from FUSE mount
- **DistributedSampler**: Shards data across all ranks (multi-node aware)
- **Persistent workers**: DataLoader keeps worker processes alive between epochs
- **DDP with NCCL**: Gradient synchronization across all GPUs

In [ ]:
def train_fn(epochs, batch_size, learning_rate, num_workers_dl, prefetch, output_dir, seed, all_paths, class_to_idx):
    import os, random, datetime, time
    import numpy as np

    import torch
    import torch.nn as nn
    import torch.optim as optim
    import torch.distributed as dist
    from torch.nn.parallel import DistributedDataParallel as DDP
    from torch.utils.data import Dataset, DataLoader
    from torch.utils.data.distributed import DistributedSampler
    from torchvision import transforms
    import torchvision.models as models
    from PIL import Image, ImageFile
    import torch.multiprocessing

    torch.multiprocessing.set_sharing_strategy("file_system")
    ImageFile.LOAD_TRUNCATED_IMAGES = True

    rank       = int(os.environ.get("RANK", 0))
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    world_size = int(os.environ.get("WORLD_SIZE", 1))

    dist.init_process_group(backend="nccl", timeout=datetime.timedelta(minutes=30))
    torch.cuda.set_device(local_rank)
    device = torch.device(f"cuda:{local_rank}")

    def log(msg):
        if rank == 0:
            print(f"[rank {rank}] {msg}", flush=True)

    log(f"world_size={world_size} local_rank={local_rank}")

    random.seed(seed + rank)
    np.random.seed(seed + rank)
    torch.manual_seed(seed + rank)
    torch.cuda.manual_seed_all(seed + rank)

    # Build labels from class map
    num_classes = len(class_to_idx)
    all_labels = [class_to_idx[p.split("/")[-2]] for p in all_paths]

    log(f"Total samples: {len(all_paths):,}, classes: {num_classes}")

    # Dataset
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    class PathDataset(Dataset):
        def __init__(self, paths, labels, transform):
            self.paths = paths
            self.labels = labels
            self.transform = transform
        def __len__(self):
            return len(self.paths)
        def __getitem__(self, idx):
            with Image.open(self.paths[idx]) as img:
                img = img.convert("RGB")
            if self.transform:
                img = self.transform(img)
            return img, self.labels[idx]

    dataset = PathDataset(all_paths, all_labels, transform=train_transform)
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)
    loader  = DataLoader(
        dataset, batch_size=batch_size, sampler=sampler,
        num_workers=num_workers_dl, pin_memory=True,
        prefetch_factor=prefetch if num_workers_dl > 0 else None,
        persistent_workers=True if num_workers_dl > 0 else False,
        drop_last=False,
    )
    log(f"Shard: {sampler.num_samples} samples/rank, {len(loader)} batches/rank")

    # Model
    model = models.resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    model = model.to(device)
    model = DDP(model, device_ids=[local_rank], output_device=local_rank)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # Training loop
    for epoch in range(epochs):
        sampler.set_epoch(epoch)
        model.train()
        running_loss, total = 0.0, 0

        for inputs, labels_batch in loader:
            inputs = inputs.to(device, non_blocking=True)
            labels_batch = labels_batch.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(inputs)
            loss = criterion(outputs, labels_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            total += inputs.size(0)

        if total > 0:
            log(f"Epoch {epoch}  Loss: {running_loss / total:.4f}")

    log(f"Training complete: {datetime.datetime.now()}")

    if rank == 0:
        os.makedirs(output_dir, exist_ok=True)
        p = os.path.join(output_dir, "model.pth")
        torch.save(model.module.state_dict(), p)
        log(f"Model saved to {p}")

    dist.destroy_process_group()
    return {"status": "success"}

## Launch Training

`TorchDistributor` spawns `NUM_PROCESSES` GPU workers per node. With `local_mode=False`, workers are distributed across all cluster nodes. Each worker trains a replica of the model with DDP synchronizing gradients after each backward pass.

In [ ]:
start_ts = dt.now()
print(f"Training started: {start_ts}")
print(f"Config: {NUM_PROCESSES} GPUs/node, LOCAL_MODE={LOCAL_MODE}, {EPOCHS} epochs, batch_size={BATCH_SIZE}")

result = TorchDistributor(
    num_processes=NUM_PROCESSES,
    local_mode=LOCAL_MODE,
    use_gpu=True,
).run(
    train_fn,
    EPOCHS, BATCH_SIZE, LEARNING_RATE, NUM_WORKERS, PREFETCH,
    OUTPUT_DIR, SEED, all_paths, class_to_idx,
)

end_ts = dt.now()
elapsed = (end_ts - start_ts).total_seconds()
print(f"\nTraining complete: {end_ts}")
print(f"Total time: {elapsed:.1f}s ({elapsed/60:.1f} min)")